In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI


In [2]:
load_dotenv()

True

In [6]:
llm=ChatGroq(model="deepseek-r1-distill-llama-70b")

In [7]:
print(llm.invoke("What is the capital of France?").content)

<think>

</think>

The capital of France is Paris.


In [8]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [9]:
embedding_model=GoogleGenerativeAIEmbeddings(model="models/embedding-001")


In [10]:
embeddings = embedding_model.embed_query("What is the capital of France?")  # Example usage

### Data Ingestion

In [25]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
import os

In [26]:
filepath = os.path.join(os.getcwd(), "data", "sample.pdf")

In [27]:
loader = PyPDFLoader(filepath)

In [28]:
doccument = loader.load()

In [30]:
len(doccument)

11

In [35]:
textsplitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

In [36]:
docs = textsplitter.split_documents(doccument)

In [37]:
len(docs)

25

In [39]:
docs[0].metadata

{'producer': 'xdvipdfmx (20210318)',
 'creator': 'LaTeX with hyperref',
 'creationdate': '2023-04-17T18:14:20-03:00',
 'author': 'Jeremy Orloff and Jonathan Bloom',
 'moddate': '2023-04-17T20:01:04-04:00',
 'title': '18.05 S22 Reading 7a: Joint Distributions, Independence',
 'source': '/Users/deepu/Downloads/study/LLMOps/document_rag/notebook/data/sample.pdf',
 'total_pages': 11,
 'page': 0,
 'page_label': '1'}

In [41]:
docs[0].page_content

'Joint Distributions, Independence \nClass 7, 18.05 \nJeremy Orloff and Jonathan Bloom \n1 Learning Goals \n1. Understand what is meant by a joint pmf, pdf and cdf of two random variables. \n2. Be able to compute probabilities and marginals from a joint pmf or pdf. \n3. Be able to test whether two random variables are independent. \n2 Introduction \nIn science and in real life, we are often interested in two (or more) random variables at the \nsame time. F or example, we might measure the height and weight of giraffes, or the IQ \nand birthweight of children, or the frequency of exercise and the rate of heart disease in \nadults, or the level of air pollution and rate of respiratory illness in cities, or the number of \nF acebook friends and the age of F acebook members. \nThink: What relationship would you expect in each of the five examples above? Why? \nIn such situations the random variables have a joint distribution that allows us to compute'

In [42]:
from langchain_community.vectorstores import FAISS

In [43]:
vectorstore = FAISS.from_documents(docs, embedding_model)

In [45]:
vectorstore.similarity_search("What is the capital of France?", k=1)

[Document(id='de43b30a-e87c-4ef2-b83f-6c8f820db423', metadata={'producer': 'xdvipdfmx (20210318)', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-04-17T18:14:20-03:00', 'author': 'Jeremy Orloff and Jonathan Bloom', 'moddate': '2023-04-17T20:01:04-04:00', 'title': '18.05 S22 Reading 7a: Joint Distributions, Independence', 'source': '/Users/deepu/Downloads/study/LLMOps/document_rag/notebook/data/sample.pdf', 'total_pages': 11, 'page': 10, 'page_label': '11'}, page_content='MIT OpenCourseWare \nhttps://ocw.mit.edu \n18.05 Introduction to Probability and Statistics \nSpring 2022 \nFor information about citing these materials or our T erms of Use, visit: https://ocw.mit.edu/terms.')]

In [53]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [54]:
retriever.invoke("Discrete case")

[Document(id='de43b30a-e87c-4ef2-b83f-6c8f820db423', metadata={'producer': 'xdvipdfmx (20210318)', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-04-17T18:14:20-03:00', 'author': 'Jeremy Orloff and Jonathan Bloom', 'moddate': '2023-04-17T20:01:04-04:00', 'title': '18.05 S22 Reading 7a: Joint Distributions, Independence', 'source': '/Users/deepu/Downloads/study/LLMOps/document_rag/notebook/data/sample.pdf', 'total_pages': 11, 'page': 10, 'page_label': '11'}, page_content='MIT OpenCourseWare \nhttps://ocw.mit.edu \n18.05 Introduction to Probability and Statistics \nSpring 2022 \nFor information about citing these materials or our T erms of Use, visit: https://ocw.mit.edu/terms.'),
 Document(id='59713a34-9810-4e37-9e6d-66a716a972cb', metadata={'producer': 'xdvipdfmx (20210318)', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-04-17T18:14:20-03:00', 'author': 'Jeremy Orloff and Jonathan Bloom', 'moddate': '2023-04-17T20:01:04-04:00', 'title': '18.05 S22 Reading 7a: Joint D

In [55]:
prompt_template = "You are a helpful assistant. You will be given a question and a set of documents. Use the documents to answer the question"

In [63]:
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [64]:
prompt_template = """
        Answer the question based on the context provided below. 
        If the context does not contain sufficient information, respond with: 
        "I do not have enough information about this."

        Context: {context}

        Question: {question}

        Answer:"""

In [65]:
prompt=PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

In [66]:
prompt

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\n        Answer the question based on the context provided below. \n        If the context does not contain sufficient information, respond with: \n        "I do not have enough information about this."\n\n        Context: {context}\n\n        Question: {question}\n\n        Answer:')

In [67]:
pareser = StrOutputParser()

In [68]:
def format_documents(docs):
    return "\n\n".join([doc.page_content for doc in docs])

In [69]:
from langchain_core.runnables import RunnablePassthrough

In [73]:
# rag_chain = prompt | retriever | llm | pareser

rag_chain = (
    { "context" : retriever | format_documents, "question": RunnablePassthrough()}
    | prompt 
    | llm 
    | pareser
)

In [74]:
rag_chain.invoke("Joint cumulative distribution function")

"<think>\nOkay, so I need to figure out the answer to the question about the joint cumulative distribution function based on the provided context. Let me read through the context again to make sure I understand it properly.\n\nThe context starts by defining the joint CDF, F(x, y), as the probability that both X is less than or equal to x and Y is less than or equal to y. That is, F(x, y) = P(X ≤ x, Y ≤ y). It then goes on to explain that for continuous random variables, the joint CDF is given by the double integral of the joint density function over the range from a to x and c to y. For discrete variables, it's a double sum of the joint PMF over all xi ≤ x and yj ≤ y.\n\nThe context also mentions that to get the joint PDF from the CDF, you take partial derivatives with respect to x and y. For discrete variables, the joint PMF is the sum of the probabilities where each xi is less than or equal to x and each yj is less than or equal to y.\n\nIn the section about events, it talks about ho